In [3]:
from ingest import build_index
from rag_helper import RAGBase
from openai import OpenAI

## Load Files

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

## Process data

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### 1. How many lesson pages

In [7]:
len(documents)

72

In [ ]:
index = build_index(documents=documents, text_fields=["content"], keyword_fields=["filename"])

answer = index.search("How does the agentic loop keep calling the model until it stops?", num_results=1)

### 2. Indexing and searching

In [7]:
answer[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [8]:
from toyaikit.llm import OpenAIClient

In [9]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()
assistant = RAGBase(index, openai_client)

In [10]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [11]:
answer

{'response': 'The loop keeps calling the model by checking whether the response contains any `function_call` items.\n\n- It sends the current `messages` history to the model.\n- If the model returns a function call, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.\n- If there are no function calls in that turn, the loop breaks.\n\nSo the stopping condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words, it keeps looping until the model returns a final message with no more tool calls.',
 'input_tokens': 7136,
 'output_tokens': 125}

### 3. RAG

### 4. Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

How many chunks do you get?

In [13]:
len(chunks)

295

Q5. RAG with chunking

In [14]:
chunk_index = build_index(documents=chunks, text_fields=["content"], keyword_fields=["filename"])
assistant = RAGBase(chunk_index, openai_client)
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [16]:
print("input_tokens:", answer["input_tokens"])
print("output_tokens:", answer["output_tokens"])

input_tokens: 2319
output_tokens: 119


Q6. Turning it into an agent


In [36]:
def search(query: str) -> dict[str, str]:
    """
    Search the lessons database for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5,
        boost_dict = {'content': 3.0, 'filename': 3}
    )

ToyAI

In [22]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [37]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [24]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [38]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# define the agent
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [39]:
prompt = "How does the agentic loop work, and how is it different from plain RAG?"

result = runner.loop(
    prompt=prompt,
    callback=callback,
)

-> Response received


-> Response received
